# Valid Plasmid Scorer — Curriculum Reward Function Testing

Tests the `ValidPlasmidScorer` which extends the plannotate BLAST scorer with
structural validity penalties:
- Excess ORIs (>1)
- Excess AMRs (>2)
- Unrequested functional elements
- Sequence length (>8 kbp)
- Promoter–CDS adjacency bonus

**Pipeline:**
1. Score real plasmids from the training set (ground truth)
2. Run inference on `McClain/PlasmidLM-kmer6-GRPO-plannotate`
3. Score generated samples
4. Compare — real plasmids should score higher

In [1]:
import sys
import os
import gc
import time
import re

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from collections import defaultdict

## 1. Load Data

In [2]:
plannotate_db = pd.read_parquet(os.path.join(PROJECT_ROOT, "data", "plannotate_db.parquet"))
motif_registry = pd.read_parquet(os.path.join(PROJECT_ROOT, "data", "motif_registry_combined.parquet"))
training_pairs = pd.read_parquet(os.path.join(PROJECT_ROOT, "data", "training_pairs_v4.parquet"))

print(f"Plannotate DB: {plannotate_db.shape[0]:,} sequences")
print(f"Motif registry: {motif_registry.shape[0]:,} entries, {motif_registry['token'].nunique()} unique tokens")
print(f"Training pairs: {training_pairs.shape[0]:,} plasmids")

Plannotate DB: 567,340 sequences
Motif registry: 660 entries, 57 unique tokens
Training pairs: 108,468 plasmids


## 2. Initialize the ValidPlasmidScorer

In [ ]:
from post_training.scorers.valid import ValidPlasmidScorer

db_dir = os.path.join(PROJECT_ROOT, "data", "plannotate_blast_db")

t0 = time.time()
scorer = ValidPlasmidScorer(
    plannotate_df=plannotate_db,
    motif_registry_df=motif_registry,
    db_dir=db_dir,
    evalue=1e-5,
    # Penalty weights
    excess_ori_penalty=0.15,
    excess_amr_penalty=0.10,
    unrequested_penalty=0.05,
    length_penalty_per_kb=0.01,
    length_soft_max=8000.0,
    length_penalty_cap=0.15,
    adjacency_bonus=0.05,
    adjacency_max_gap=500,
    broad_min_coverage=80.0,
    broad_min_identity=95.0,
)
print(f"Scorer initialized in {time.time() - t0:.1f}s")
print(f"Token bridge: {len(scorer._base._token_bridge)} tokens")
print(f"Full reverse map: {len(scorer._all_sseqid_to_token)} sseqid entries")
print(f"Broad thresholds: coverage>={scorer.broad_min_coverage:.0f}%, identity>={scorer.broad_min_identity:.0f}%")

## 3. Sample Real Plasmids

In [ ]:
HARD_PREFIXES = {"AMR_", "ORI_", "PROM_", "REPORTER_", "REP_", "TAG_", "ELEM_"}

def count_hard_tokens(prompt):
    tokens = re.findall(r"<[^>]+>", prompt)
    return sum(1 for t in tokens if any(t.strip("<>").startswith(p) for p in HARD_PREFIXES))

# Filter to plasmids with 2+ hard tokens and reasonable length
mask = (
    training_pairs["prompt"].apply(count_hard_tokens) >= 2
) & (
    training_pairs["sequence"].str.len().between(2000, 15000)
)
candidates = training_pairs[mask]
print(f"Candidates with 2+ hard tokens and 2-15kbp: {len(candidates):,}")

N_SAMPLE = 50
sample = candidates.sample(N_SAMPLE, random_state=42).reset_index(drop=True)
print(f"Sampled {len(sample)} plasmids")
print(f"Length range: {sample['sequence'].str.len().min():,} – {sample['sequence'].str.len().max():,} bp")
sample[["prompt", "sequence"]].head()

## 4. Score Real Plasmids

In [ ]:
real_results = []

t0 = time.time()
for i, row in sample.iterrows():
    result = scorer.score_sequence_detailed(row["prompt"], row["sequence"])
    result["idx"] = i
    result["prompt"] = row["prompt"]
    real_results.append(result)
    if (i + 1) % 10 == 0:
        print(f"  Scored {i + 1}/{N_SAMPLE}")

print(f"Scored {len(real_results)} real plasmids in {time.time() - t0:.1f}s")

real_rewards = [r["reward"] for r in real_results]
real_base = [r["base_reward"] for r in real_results]
print(f"\nReal plasmid rewards:")
print(f"  Final  — mean={np.mean(real_rewards):.3f}, median={np.median(real_rewards):.3f}")
print(f"  Base   — mean={np.mean(real_base):.3f}, median={np.median(real_base):.3f}")

### 4a. Penalty Breakdown on Real Plasmids

In [ ]:
penalty_df = pd.DataFrame([
    {
        "reward": r["reward"],
        "base_reward": r["base_reward"],
        "multiplier": r["structural_multiplier"],
        "ori_count": r["ori_count"],
        "ori_penalty": r["ori_penalty"],
        "amr_count": r["amr_count"],
        "amr_penalty": r["amr_penalty"],
        "unrequested_count": len(r["unrequested_tokens"]),
        "unrequested_penalty": r["unrequested_penalty"],
        "seq_len": r["seq_len"],
        "length_penalty": r["length_penalty"],
        "adjacency_bonus": r["adjacency_bonus"],
        "n_adj_pairs": len(r["adjacency_pairs"]),
    }
    for r in real_results
])

print("Penalty/bonus statistics on real plasmids:")
print(penalty_df[["multiplier", "ori_penalty", "amr_penalty",
                   "unrequested_penalty", "length_penalty", "adjacency_bonus"]].describe().round(3))
print(f"\nORI counts: {penalty_df['ori_count'].value_counts().sort_index().to_dict()}")
print(f"AMR counts: {penalty_df['amr_count'].value_counts().sort_index().to_dict()}")
print(f"Unrequested elements: mean={penalty_df['unrequested_count'].mean():.1f}, max={penalty_df['unrequested_count'].max()}")
print(f"Adjacency pairs found: {penalty_df['n_adj_pairs'].sum()} across {(penalty_df['n_adj_pairs'] > 0).sum()} plasmids")

## 5. Generate Sequences from Model

Runs inference on `McClain/PlasmidLM-kmer6-GRPO-plannotate` using the same
prompts as the real plasmids. **Requires GPU.**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor
from tqdm import tqdm
import hashlib
import json


class SanitizeLogitsProcessor(LogitsProcessor):
    """Clamp NaN/inf logits before sampling to avoid CUDA errors."""

    def __call__(self, input_ids, scores):
        bad = scores.isnan() | scores.isinf() | (scores < -1e9)
        if bad.any():
            scores = scores.masked_fill(bad, -1e4)
        return scores


MODEL_ID = "McClain/PlasmidLM-kmer6-GRPO-plannotate"
MAX_NEW_TOKENS = 1500
TEMPERATURE = 0.2
N_CANDIDATES = 4
CACHE_DIR = os.path.join(PROJECT_ROOT, "eval", "scorers", "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

prompt_digest = hashlib.sha1("\n".join(sample["prompt"].tolist()).encode()).hexdigest()[:12]
cache_path = os.path.join(
    CACHE_DIR,
    f"valid_generations_{MODEL_ID.split('/')[-1]}_n{N_SAMPLE}_k{N_CANDIDATES}_t{str(TEMPERATURE).replace('.', 'p')}_m{MAX_NEW_TOKENS}_{prompt_digest}.json",
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Cache path: {cache_path}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = "<PAD>"

logits_sanitizer = SanitizeLogitsProcessor()
load_model_for_generation = not os.path.exists(cache_path)
if load_model_for_generation:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        dtype=torch.float32,
        device_map="auto" if device == "cuda" else None,
    )
    model.eval()
    print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
else:
    model = None
    print("Using cached generations; model load skipped.")

In [ ]:
def extract_dna(text: str) -> str:
    seq = re.sub(r"<[^>]+>", "", text.upper())
    return re.sub(r"[^ATGCN]", "", seq)


prompts_raw = sample["prompt"].tolist()

torch.manual_seed(42)
t0 = time.time()

if os.path.exists(cache_path):
    with open(cache_path) as f:
        cache_payload = json.load(f)
    generated_rollouts = cache_payload["generated_rollouts"]
    print(f"Loaded cached generations for {len(generated_rollouts)} prompts in {time.time() - t0:.1f}s")
else:
    generated_rollouts = []
    for prompt in tqdm(prompts_raw, desc="Generating prompts"):
        prompt_rollouts = []
        for rollout_idx in range(N_CANDIDATES):
            encoded = tokenizer(prompt, return_tensors="pt")
            if device == "cuda":
                encoded = encoded.to(device)

            with torch.no_grad():
                output_ids = model.generate(
                    input_ids=encoded["input_ids"],
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                    do_sample=True,
                    top_k=50,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=1,
                    logits_processor=[logits_sanitizer],
                )

            prompt_len = encoded["input_ids"].shape[1]
            gen_ids = output_ids[0][prompt_len:]
            gen_ids = gen_ids[gen_ids != tokenizer.pad_token_id]
            prompt_rollouts.append(tokenizer.decode(gen_ids.tolist()))
        generated_rollouts.append(prompt_rollouts)

    with open(cache_path, "w") as f:
        json.dump(
            {
                "model_id": MODEL_ID,
                "temperature": TEMPERATURE,
                "n_candidates": N_CANDIDATES,
                "max_new_tokens": MAX_NEW_TOKENS,
                "prompt_digest": prompt_digest,
                "generated_rollouts": generated_rollouts,
            },
            f,
        )
    print(f"Saved generations to cache: {cache_path}")

elapsed = time.time() - t0
print(f"Prepared {len(generated_rollouts)} prompt batches in {elapsed:.1f}s")
print(f"Total generated sequences: {sum(len(x) for x in generated_rollouts)}")

# Clean up GPU memory
if model is not None:
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

gen_dna = [extract_dna(t) for prompt_rollouts in generated_rollouts for t in prompt_rollouts]
gen_lens = [len(s) for s in gen_dna]
print(f"Generated DNA lengths across all rollouts: min={min(gen_lens):,}, median={int(np.median(gen_lens)):,}, max={max(gen_lens):,}")

## 6. Score Generated Sequences

In [ ]:
gen_rollout_results = []

t0 = time.time()
for i, (prompt, prompt_rollouts) in enumerate(zip(prompts_raw, generated_rollouts)):
    prompt_results = []
    for rollout_idx, gen_seq in enumerate(prompt_rollouts):
        result = scorer.score_sequence_detailed(prompt, gen_seq)
        result["idx"] = i
        result["rollout_idx"] = rollout_idx
        result["prompt"] = prompt
        prompt_results.append(result)
    gen_rollout_results.append(prompt_results)
    if (i + 1) % 10 == 0:
        print(f"  Scored {i + 1}/{len(prompts_raw)} prompts")

print(f"Scored {sum(len(x) for x in gen_rollout_results)} generated sequences in {time.time() - t0:.1f}s")

gen_results = [max(prompt_results, key=lambda r: r["reward"]) for prompt_results in gen_rollout_results]
gen_best_rewards = [r["reward"] for r in gen_results]
gen_best_base = [r["base_reward"] for r in gen_results]
gen_mean_rewards = [np.mean([r["reward"] for r in prompt_results]) for prompt_results in gen_rollout_results]
gen_mean_base = [np.mean([r["base_reward"] for r in prompt_results]) for prompt_results in gen_rollout_results]
gen_rollout_rewards = [r["reward"] for prompt_results in gen_rollout_results for r in prompt_results]
gen_rollout_base = [r["base_reward"] for prompt_results in gen_rollout_results for r in prompt_results]

gen_rewards = gen_best_rewards
gen_base = gen_best_base

print(f"\nGenerated sequence rewards (best-of-{N_CANDIDATES}):")
print(f"  Final  — mean={np.mean(gen_best_rewards):.3f}, median={np.median(gen_best_rewards):.3f}")
print(f"  Base   — mean={np.mean(gen_best_base):.3f}, median={np.median(gen_best_base):.3f}")
print(f"\nGenerated sequence rewards (mean over {N_CANDIDATES} rollouts per prompt):")
print(f"  Final  — mean={np.mean(gen_mean_rewards):.3f}, median={np.median(gen_mean_rewards):.3f}")
print(f"  Base   — mean={np.mean(gen_mean_base):.3f}, median={np.median(gen_mean_base):.3f}")
print(f"\nGenerated sequence rewards (all rollouts pooled):")
print(f"  Final  — mean={np.mean(gen_rollout_rewards):.3f}, median={np.median(gen_rollout_rewards):.3f}")
print(f"  Base   — mean={np.mean(gen_rollout_base):.3f}, median={np.median(gen_rollout_base):.3f}")

## 7. Compare Real vs Generated

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Final reward
axes[0].hist(real_rewards, bins=20, alpha=0.6, label="Real", color="#2196F3")
axes[0].hist(gen_rollout_rewards, bins=20, alpha=0.35, label="Generated (all rollouts)", color="#FFAB91")
axes[0].hist(gen_mean_rewards, bins=20, alpha=0.5, label=f"Generated mean-of-{N_CANDIDATES}", color="#FF9800")
axes[0].hist(gen_best_rewards, bins=20, alpha=0.6, label=f"Generated best-of-{N_CANDIDATES}", color="#FF5722")
axes[0].set_xlabel("Final Reward")
axes[0].set_title("Final Reward (base × structural)")
axes[0].legend()
axes[0].axvline(np.mean(real_rewards), color="#2196F3", ls="--", lw=1)
axes[0].axvline(np.mean(gen_best_rewards), color="#FF5722", ls="--", lw=1)

# Base reward (plannotate only)
axes[1].hist(real_base, bins=20, alpha=0.6, label="Real", color="#2196F3")
axes[1].hist(gen_rollout_base, bins=20, alpha=0.35, label="Generated (all rollouts)", color="#FFAB91")
axes[1].hist(gen_mean_base, bins=20, alpha=0.5, label=f"Generated mean-of-{N_CANDIDATES}", color="#FF9800")
axes[1].hist(gen_best_base, bins=20, alpha=0.6, label=f"Generated best-of-{N_CANDIDATES}", color="#FF5722")
axes[1].set_xlabel("Base Plannotate Reward")
axes[1].set_title("Base Plannotate Reward")
axes[1].legend()

# Structural multiplier (best rollout per prompt)
real_mult = [r["structural_multiplier"] for r in real_results]
gen_mult = [r["structural_multiplier"] for r in gen_results]
axes[2].hist(real_mult, bins=20, alpha=0.6, label="Real", color="#2196F3")
axes[2].hist(gen_mult, bins=20, alpha=0.6, label=f"Generated best-of-{N_CANDIDATES}", color="#FF5722")
axes[2].set_xlabel("Structural Multiplier")
axes[2].set_title("Structural Multiplier")
axes[2].axvline(1.0, color="gray", ls=":", lw=1)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
data = [
    real_rewards,
    gen_rollout_rewards,
    gen_mean_rewards,
    gen_best_rewards,
    real_base,
    gen_best_base,
]
labels = [
    "Real\n(final)",
    "Generated\n(all rollouts)",
    f"Generated\n(mean-of-{N_CANDIDATES})",
    f"Generated\n(best-of-{N_CANDIDATES})",
    "Real\n(base)",
    f"Generated\nbase best-of-{N_CANDIDATES}",
]
bp = ax.boxplot(data, labels=labels, patch_artist=True)
colors = ["#2196F3", "#FFCCBC", "#FFB74D", "#FF5722", "#90CAF9", "#FFAB91"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Reward")
ax.set_title("Real vs Generated — Reward Comparison")
ax.axhline(0, color="gray", ls=":", lw=0.5)
plt.tight_layout()
plt.show()

## 8. Penalty Breakdown Comparison

In [ ]:
def make_penalty_df(results, label):
    rows = []
    for r in results:
        rows.append({
            "source": label,
            "reward": r["reward"],
            "base_reward": r["base_reward"],
            "multiplier": r["structural_multiplier"],
            "ori_count": r["ori_count"],
            "ori_penalty": r["ori_penalty"],
            "amr_count": r["amr_count"],
            "amr_penalty": r["amr_penalty"],
            "unrequested_count": len(r["unrequested_tokens"]),
            "unrequested_penalty": r["unrequested_penalty"],
            "seq_len": r["seq_len"],
            "length_penalty": r["length_penalty"],
            "adjacency_bonus": r["adjacency_bonus"],
        })
    return pd.DataFrame(rows)

df_real = make_penalty_df(real_results, "Real")
df_gen = make_penalty_df(gen_results, "Generated")
df_all = pd.concat([df_real, df_gen], ignore_index=True)

print("Mean penalties/bonuses:")
print(df_all.groupby("source")[
    ["ori_penalty", "amr_penalty", "unrequested_penalty",
     "length_penalty", "adjacency_bonus", "multiplier"]
].mean().round(4).T)

In [ ]:
penalty_cols = ["ori_penalty", "amr_penalty", "unrequested_penalty", "length_penalty", "adjacency_bonus"]
means = df_all.groupby("source")[penalty_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(penalty_cols))
w = 0.35
ax.bar(x - w/2, means.loc["Real"], w, label="Real", color="#2196F3", alpha=0.8)
ax.bar(x + w/2, means.loc["Generated"], w, label="Generated", color="#FF5722", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", " ").title() for c in penalty_cols], rotation=15)
ax.set_ylabel("Mean Value")
ax.set_title("Penalty/Bonus Breakdown: Real vs Generated")
ax.legend()
ax.axhline(0, color="gray", ls=":", lw=0.5)
plt.tight_layout()
plt.show()

## 9. Per-Sample Diagnostic

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Paired comparison: real vs gen reward for same prompt
axes[0].scatter(real_rewards, gen_rewards, alpha=0.6, s=40)
lim = [0, max(max(real_rewards), max(gen_rewards)) * 1.05]
axes[0].plot(lim, lim, "k--", lw=0.8, alpha=0.5)
axes[0].set_xlabel("Real Plasmid Reward")
axes[0].set_ylabel("Generated Reward")
axes[0].set_title("Paired: Real vs Generated (same prompt)")

# Delta (real - generated)
deltas = [r - g for r, g in zip(real_rewards, gen_rewards)]
axes[1].hist(deltas, bins=20, color="#4CAF50", alpha=0.7)
axes[1].axvline(0, color="red", ls="--", lw=1)
axes[1].axvline(np.mean(deltas), color="black", ls="--", lw=1, label=f"mean={np.mean(deltas):.3f}")
axes[1].set_xlabel("Real − Generated Reward")
axes[1].set_title("Reward Delta (positive = real is better)")
axes[1].legend()

plt.tight_layout()
plt.show()

pct_real_better = sum(1 for d in deltas if d > 0) / len(deltas) * 100
print(f"Real plasmid scores higher: {pct_real_better:.0f}% of the time")

## 10. Inspect Worst-Scoring Generated Sequences

Look at which penalties are hitting hardest on the generated sequences.

In [ ]:
gen_sorted = sorted(enumerate(gen_results), key=lambda x: x[1]["reward"])

print("=== 5 WORST generated sequences ===")
for rank, (idx, r) in enumerate(gen_sorted[:5]):
    print(f"\n--- #{rank+1} (idx={idx}) reward={r['reward']:.3f} ---")
    print(f"  Prompt: {r['prompt']}")
    print(f"  Base reward: {r['base_reward']:.3f}")
    print(f"  Multiplier:  {r['structural_multiplier']:.3f}")
    print(f"  ORI count:   {r['ori_count']}  (penalty={r['ori_penalty']:.3f})")
    print(f"  AMR count:   {r['amr_count']}  (penalty={r['amr_penalty']:.3f})")
    print(f"  Unrequested: {r['unrequested_tokens']}  (penalty={r['unrequested_penalty']:.3f})")
    print(f"  Seq len:     {r['seq_len']:,} bp  (penalty={r['length_penalty']:.3f})")
    print(f"  Adjacency:   {r['adjacency_pairs']}  (bonus={r['adjacency_bonus']:.3f})")
    print(f"  All found:   {r['found_tokens']}")

print("\n\n=== 5 BEST generated sequences ===")
for rank, (idx, r) in enumerate(gen_sorted[-5:]):
    print(f"\n--- #{rank+1} (idx={idx}) reward={r['reward']:.3f} ---")
    print(f"  Prompt: {r['prompt']}")
    print(f"  Base reward: {r['base_reward']:.3f}")
    print(f"  Multiplier:  {r['structural_multiplier']:.3f}")
    print(f"  Found:       {r['found_tokens']}")

## Summary

## 11. Rollout Diagnostics

Use these cells for GRPO-facing analysis. They summarize reward signal at three levels:
- pooled rollout rewards
- mean reward across 4 rollouts per prompt
- best reward across 4 rollouts per prompt

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(real_rewards, gen_mean_rewards, alpha=0.45, s=35, color="#FF9800", label=f"Generated mean-of-{N_CANDIDATES}")
axes[0].scatter(real_rewards, gen_best_rewards, alpha=0.6, s=35, color="#FF5722", label=f"Generated best-of-{N_CANDIDATES}")
lim = [0, max(max(real_rewards), max(gen_best_rewards)) * 1.05]
axes[0].plot(lim, lim, "k--", lw=0.8, alpha=0.5)
axes[0].set_xlabel("Real Plasmid Reward")
axes[0].set_ylabel("Generated Reward")
axes[0].set_title("Paired: Real vs Generated (same prompt)")
axes[0].legend()

deltas_best = [r - g for r, g in zip(real_rewards, gen_best_rewards)]
deltas_mean = [r - g for r, g in zip(real_rewards, gen_mean_rewards)]
axes[1].hist(deltas_mean, bins=20, color="#FFB74D", alpha=0.55, label=f"Real - mean-of-{N_CANDIDATES}")
axes[1].hist(deltas_best, bins=20, color="#4CAF50", alpha=0.65, label=f"Real - best-of-{N_CANDIDATES}")
axes[1].axvline(0, color="red", ls="--", lw=1)
axes[1].axvline(np.mean(deltas_best), color="black", ls="--", lw=1, label=f"best mean={np.mean(deltas_best):.3f}")
axes[1].set_xlabel("Real - Generated Reward")
axes[1].set_title("Reward Delta (positive = real is better)")
axes[1].legend()

plt.tight_layout()
plt.show()

pct_real_better_best = sum(1 for d in deltas_best if d > 0) / len(deltas_best) * 100
pct_real_better_mean = sum(1 for d in deltas_mean if d > 0) / len(deltas_mean) * 100
print(f"Real plasmid scores higher than best-of-{N_CANDIDATES}: {pct_real_better_best:.0f}% of the time")
print(f"Real plasmid scores higher than mean-of-{N_CANDIDATES}: {pct_real_better_mean:.0f}% of the time")

In [ ]:
print("=" * 72)
print("ValidPlasmidScorer — Rollout Summary")
print("=" * 72)
print(f"Model: {MODEL_ID}")
print(f"Temperature: {TEMPERATURE}")
print(f"Rollouts per prompt: {N_CANDIDATES}")
print(f"Cached generations: {os.path.exists(cache_path)}")

print(f"\n{'':>30} {'Real':>10} {'Gen mean':>10} {'Gen best':>10}")
print(f"{'Final reward (mean)':>30} {np.mean(real_rewards):>10.3f} {np.mean(gen_mean_rewards):>10.3f} {np.mean(gen_best_rewards):>10.3f}")
print(f"{'Final reward (median)':>30} {np.median(real_rewards):>10.3f} {np.median(gen_mean_rewards):>10.3f} {np.median(gen_best_rewards):>10.3f}")
print(f"{'Base reward (mean)':>30} {np.mean(real_base):>10.3f} {np.mean(gen_mean_base):>10.3f} {np.mean(gen_best_base):>10.3f}")
print(f"{'Rollout reward (pooled mean)':>30} {'-':>10} {np.mean(gen_rollout_rewards):>10.3f} {'-':>10}")
print(f"{'Rollout reward (pooled median)':>30} {'-':>10} {np.median(gen_rollout_rewards):>10.3f} {'-':>10}")

print(f"\n{'':>30} {'Real':>10} {'Gen best':>10}")
print(f"{'Structural mult (mean)':>30} {np.mean(real_mult):>10.3f} {np.mean([r['structural_multiplier'] for r in gen_results]):>10.3f}")
print(f"{'Mean ORI count':>30} {df_real['ori_count'].mean():>10.1f} {df_gen['ori_count'].mean():>10.1f}")
print(f"{'Mean AMR count':>30} {df_real['amr_count'].mean():>10.1f} {df_gen['amr_count'].mean():>10.1f}")
print(f"{'Mean unrequested':>30} {df_real['unrequested_count'].mean():>10.1f} {df_gen['unrequested_count'].mean():>10.1f}")
print(f"{'Mean seq len (bp)':>30} {df_real['seq_len'].mean():>10.0f} {df_gen['seq_len'].mean():>10.0f}")

print(f"\nReal > mean-of-{N_CANDIDATES}: {pct_real_better_mean:.0f}% of prompts")
print(f"Real > best-of-{N_CANDIDATES}: {pct_real_better_best:.0f}% of prompts")
print(f"Mean delta (real - gen mean): {np.mean(deltas_mean):.3f}")
print(f"Mean delta (real - gen best): {np.mean(deltas_best):.3f}")

In [ ]:
print("=" * 60)
print("ValidPlasmidScorer — Summary")
print("=" * 60)
print(f"\n{'':>25} {'Real':>10} {'Generated':>10}")
print(f"{'Final reward (mean)':>25} {np.mean(real_rewards):>10.3f} {np.mean(gen_rewards):>10.3f}")
print(f"{'Final reward (median)':>25} {np.median(real_rewards):>10.3f} {np.median(gen_rewards):>10.3f}")
print(f"{'Base reward (mean)':>25} {np.mean(real_base):>10.3f} {np.mean(gen_base):>10.3f}")
print(f"{'Structural mult (mean)':>25} {np.mean(real_mult):>10.3f} {np.mean(gen_mult):>10.3f}")
print(f"{'Mean ORI count':>25} {df_real['ori_count'].mean():>10.1f} {df_gen['ori_count'].mean():>10.1f}")
print(f"{'Mean AMR count':>25} {df_real['amr_count'].mean():>10.1f} {df_gen['amr_count'].mean():>10.1f}")
print(f"{'Mean unrequested':>25} {df_real['unrequested_count'].mean():>10.1f} {df_gen['unrequested_count'].mean():>10.1f}")
print(f"{'Mean seq len (bp)':>25} {df_real['seq_len'].mean():>10.0f} {df_gen['seq_len'].mean():>10.0f}")
print(f"\nReal scores higher: {pct_real_better:.0f}% of paired comparisons")